In [ ]:
# @title Setup smoke test (run this once at home before the tutorial!)
try:
    !pip install -q -r https://raw.githubusercontent.com/<org>/ccnss2026-neural-data-analysis/main/requirements.txt
    from allensdk.brain_observatory.ecephys.ecephys_project_cache import EcephysProjectCache  # noqa
    import ssm  # noqa
    print("✅ Setup complete.")
except Exception as e:
    print("❌ Setup FAILED:", e)
    raise

# CCNSS 2026 — Session 2: Dynamics and States

**Theme:** From population activity to dynamics and discrete states.

Three modules on the NLB MC_Maze dataset:
- **2A** — PCA on population activity
- **2B** — Linear Dynamical Systems (`ssm`)
- **2C** — Hidden Markov Models (`ssm`)
- **Wrap-up** — Where the field is going (LFADS, CEBRA, MARBLE)

In [ ]:
# @title Setup (run once)
!pip install -q -r https://raw.githubusercontent.com/<org>/ccnss2026-neural-data-analysis/main/requirements.txt 2>&1 | tail -5
!git clone -q https://github.com/<org>/ccnss2026-neural-data-analysis.git || (cd ccnss2026-neural-data-analysis && git pull -q)
import sys; sys.path.insert(0, "/content/ccnss2026-neural-data-analysis")

import numpy as np, matplotlib.pyplot as plt
from ccnss_helpers import data, plotting, save_checkpoint, load_or_compute
print("✅ Setup complete.")

In [ ]:
mm = data.load_mc_maze_small()
binned = mm["binned_spikes"]   # (trials, bins, neurons)
dirs = mm["trial_directions"]  # (trials,)
hand = mm["hand_trajectory"]   # (trials, bins, 2)
print(binned.shape, "directions:", np.unique(dirs))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for d in np.unique(dirs):
    m = dirs == d
    axes[0].plot(hand[m, :, 0].T, hand[m, :, 1].T, color=plt.cm.hsv(d / 8), alpha=0.3)
axes[0].set_title("Hand trajectories by reach direction"); axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
# Raster of one trial.
trial0_spikes = {n: np.where(binned[0, :, n] > 0)[0] * mm["bin_size_s"] for n in range(20)}
plotting.plot_raster(trial0_spikes, ax=axes[1])
axes[1].set_title("Trial 0 raster (first 20 neurons)");

## Module 2A — PCA on population activity

We trial-average per reach direction, stack as (conditions × time) × neurons, fit PCA,
and visualize the top 3 PC trajectories.

**Key idea:** if all neurons fired independently, you'd need N dimensions. The fact that
just 3 capture most of the variance means the population lives on a low-dimensional manifold.

In [ ]:
def trial_average_per_direction(binned, dirs):
    """Returns (n_directions, n_bins, n_neurons) average rates."""
    unique_dirs = np.unique(dirs)
    return np.stack([binned[dirs == d].mean(axis=0) for d in unique_dirs])

avg = trial_average_per_direction(binned, dirs)   # (n_dir, n_bins, n_neurons)
print(avg.shape)

In [ ]:
# EXERCISE: fit PCA on the trial-averaged data and project to the top components
# YOUR CODE HERE
raise NotImplementedError('fit PCA on the trial-averaged data and project to the top components')


In [ ]:
plotting.plot_latents_3d(Z[:, :, :3])
plt.suptitle("Top 3 PCs across reach directions");

In [ ]:
# EXERCISE: plot cumulative variance explained; how many PCs reach 80%?
# YOUR CODE HERE
raise NotImplementedError('plot cumulative variance explained; how many PCs reach 80%?')


In [ ]:
# EXERCISE: (challenge) project single trials into the trial-averaged PCA basis
# YOUR CODE HERE
raise NotImplementedError('(challenge) project single trials into the trial-averaged PCA basis')


In [ ]:
save_checkpoint("module_2A", avg=avg, Z=Z, evr=pca.explained_variance_ratio_,
                components=pca.components_, mean=pca.mean_)
print("✅ 2A checkpoint saved.")

## Module 2B — Linear Dynamical Systems

PCA gives us a low-D snapshot but ignores time. An LDS adds a transition model:

$$z_{t+1} = A z_t + \epsilon_t, \quad y_t = C z_t + \nu_t$$

We fit one with Linderman's `ssm` and compare latents to PCA. Then simulate from the fit.

> **Aside on GPFA:** structurally similar but puts a Gaussian Process prior on latents → smoother trajectories. Same conceptual goal; different smoothness regularizer.

In [ ]:
# 2-minute tour of ssm: fit a tiny 2D LDS on toy data so the API is familiar.
import ssm
toy_T, toy_D, toy_N = 200, 2, 10
toy_lds = ssm.LDS(N=toy_N, D=toy_D, emissions="poisson")
toy_y, toy_z = toy_lds.sample(T=toy_T)
toy_fit = ssm.LDS(N=toy_N, D=toy_D, emissions="poisson")
toy_fit.fit(toy_y, num_iters=20, verbose=0)
print("Toy LDS fitted successfully. API: .sample(), .fit(), .most_likely_states() etc.")

In [ ]:
# EXERCISE: fit a D=6 LDS to single-trial spike counts
# YOUR CODE HERE
raise NotImplementedError('fit a D=6 LDS to single-trial spike counts')


In [ ]:
latents = np.stack([lds.smooth(t) for t in trials])  # (trials, bins, D)
avg_latents = trial_average_per_direction(latents, dirs)   # (n_dir, bins, D)
plotting.plot_latents_3d(avg_latents[:, :, :3])
plt.suptitle("LDS latents (top 3 dims), trial-averaged per direction");

In [ ]:
# EXERCISE: simulate from the fitted LDS and compare PSTH to the real data
# YOUR CODE HERE
raise NotImplementedError('simulate from the fitted LDS and compare PSTH to the real data')


In [ ]:
# EXERCISE: (challenge) vary D, plot held-out log-likelihood
# YOUR CODE HERE
raise NotImplementedError('(challenge) vary D, plot held-out log-likelihood')


In [ ]:
save_checkpoint("module_2B", latents=latents, avg_latents=avg_latents)
print("✅ 2B checkpoint saved.")

## Module 2C — Hidden Markov Models

Instead of continuous latents, we now ask: are there **discrete states** (e.g. baseline,
preparation, movement, hold)? Fit a Poisson-HMM with `ssm` and visualize the state sequence aligned to movement onset.

In [ ]:
# EXERCISE: fit a 4-state Poisson HMM
# YOUR CODE HERE
raise NotImplementedError('fit a 4-state Poisson HMM')


In [ ]:
states = np.stack([hmm.most_likely_states(t) for t in trials])
fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(states, aspect="auto", cmap="tab10", interpolation="nearest")
ax.set_xlabel("Time bin"); ax.set_ylabel("Trial")
ax.set_title(f"Most-likely HMM state sequence (K={K})");

In [ ]:
# EXERCISE: refit with K=3 and K=6; interpret what the states correspond to
# YOUR CODE HERE
raise NotImplementedError('refit with K=3 and K=6; interpret what the states correspond to')


In [ ]:
# EXERCISE: (challenge) SLDS (Switching LDS) = 2B + 2C unified
# YOUR CODE HERE
raise NotImplementedError('(challenge) SLDS (Switching LDS) = 2B + 2C unified')


## Where the field is going

- **LFADS** — sequential VAE for neural dynamics. Pandarinath et al. 2018.
- **CEBRA** — contrastive embeddings constrained by behavior. Schneider et al. 2023.
- **MARBLE** — geometric/manifold methods for neural manifolds. Gosztolai et al. 2024.

Code-free — but worth bookmarking. See slide for paper + GitHub links.

In [ ]:
# 🎉 End of Session 2
print("Session 2 complete! Resources and further reading in the slide deck.")